In [86]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [87]:
transactions = pd.read_csv("../data/processed/enriched_transactions.csv")

transactions.shape

(32439, 14)

In [88]:
transactions.describe()

,amount,year,month
count,32439.000000,32439.000000,32439.000000
mean,4234.404242,2025.468726,6.307346
std,30739.716901,0.499029,3.198230
min,-94582.300000,2025.000000,1.000000
25%,-3668.920000,2025.000000,4.000000
50%,-1308.550000,2025.000000,6.000000
75%,-418.860000,2026.000000,9.000000
max,142866.310000,2026.000000,12.000000


In [89]:
transactions["validation_issues"].value_counts()

validation_issues
Exact duplicate rows    2
Missing descriptions    1
Missing category        1
Name: count, dtype: int64

In [90]:
transactions.columns.tolist()

['transaction_id',
 'customer_id',
 'date',
 'description',
 'amount',
 'category',
 'transaction_type',
 'merchant',
 'currency',
 'validation_issues',
 'year',
 'month',
 'day_of_week',
 'is_weekend']

In [91]:
transactions.dtypes

transaction_id           str
customer_id              str
date                     str
description              str
amount               float64
category                 str
transaction_type         str
merchant                 str
currency                 str
validation_issues        str
year                   int64
month                  int64
day_of_week              str
is_weekend              bool
dtype: object

In [92]:
print("Currency distribution:")
print(transactions["currency"].value_counts(dropna=False))

print("\nTransaction types and amount ranges:")
display(
    transactions.groupby("transaction_type")["amount"]
    .agg(["count", "min", "median", "max"])
    .round(2)
)

Currency distribution:
currency
INR    32439
Name: count, dtype: int64

Transaction types and amount ranges:


,count,min,median,max
transaction_type,,,,
credit,5070,649.0,65529.00,142866.31
debit,27369,-94582.3,-1764.83,-24.99


In [93]:
transactions["amount_magnitude"] = transactions["amount"].abs()

transactions.groupby("transaction_type")["amount_magnitude"].describe().round(2)


,count,mean,std,min,25%,50%,75%,max
transaction_type,,,,,,,,
credit,5070.0,65688.13,17194.7,649.00,54189.83,65529.00,77794.76,142866.31
debit,27369.0,7149.66,15362.3,24.99,757.40,1764.83,4689.00,94582.30


# My notes: `amount_magnitude` and why I'm using two Isolation Forest models

## 1. Why I created `amount_magnitude`

In my transaction data, I already have an `amount` column. During preprocessing, I decided to keep credits positive and debits negative:

- Credit of ₹2,000 → `amount = 2000`
- Debit of ₹850 → `amount = -850`

The sign tells me the direction of the money. Positive means money came in, negative means money went out.

For my first anomaly detection experiment, I want to look at the **size** of a transaction without its direction getting mixed into that feature. So I made a new column:

```python
transactions["amount_magnitude"] = transactions["amount"].abs()
```

`.abs()` removes the negative sign from negative numbers but leaves positive numbers as they are:

| Original `amount` | `amount_magnitude` |
|---:|---:|
| `2000` | `2000` |
| `-850` | `850` |
| `-3500` | `3500` |
| `95000` | `95000` |

I'm creating a **new feature**, not replacing my original `amount` column. I still need the signed amount later to know whether the transaction was a credit or a debit.

The reason for doing this is simple: `amount_magnitude` represents how large the transaction is in rupees, regardless of direction. It is not a high-pass/low-pass filter, and it does not set a threshold. It just transforms the data into a useful numerical feature.

Isolation Forest can then look for magnitudes that are unusually easy to isolate from the other amounts it sees. It might flag an unusually large amount, but it can also flag a very small amount if that value looks unusual in the group. It does not use a fixed rule like “anything above ₹50,000 is an anomaly.”

## 2. What I noticed in my dataset

I checked the magnitude distributions separately for credits and debits. They are quite different:

| Statistic | Credits | Debits |
|---|---:|---:|
| Transaction count | 5,070 | 27,369 |
| Median magnitude | ₹65,529 | ₹1,764.83 |
| 75th percentile | ₹77,794.76 | ₹4,689 |
| Maximum magnitude | ₹142,866.31 | ₹94,582.30 |

Credits generally have much larger amounts than debits in this dataset. There are also many more debits than credits.

That matters because a ₹50,000 transaction can have a different meaning depending on whether it's a credit or a debit. If I trained one model across both types, the different distributions could affect the anomalies it identifies.

## 3. Why I decided to use two Isolation Forest models

I decided to use **two separate models** for the first version:

1. One Isolation Forest trained only on credit transactions.
2. One Isolation Forest trained only on debit transactions.

The flow is:

```text
All transactions
       |
       +---- Select credits ----> amount_magnitude ----> Credit Isolation Forest
       |
       +---- Select debits  ----> amount_magnitude ----> Debit Isolation Forest
```

I'm choosing this approach because it gives each model its own transaction-type group to learn from. The credit model looks for unusual credit magnitudes compared with other credits; the debit model looks for unusual debit magnitudes compared with other debits.

I don't need to encode `transaction_type` as a model feature for this baseline, because I'm using it to split the data before training. Each model sees only one transaction type.

This is a design choice for my first experiment, not a claim that two models are always better. I could also test one combined model later, but I want to start with a straightforward baseline that is easier to understand and evaluate.

## 4. What this first model can and cannot tell me

Because my first model uses only `amount_magnitude`, it can flag transaction sizes that look unusual relative to the other transactions of the same type.

It **cannot yet** tell whether an amount is unusual for a particular customer. For example, a ₹40,000 debit might be normal for one customer but very unusual for another. The current feature does not include that customer's usual transaction behaviour.

It also doesn't know whether a transaction is fraudulent. An anomaly is a transaction worth investigating, not proof of fraud.

Later, I can experiment with features such as a transaction's difference from the customer's usual amount or how many transactions happen within a short time window. I don't need to add everything immediately; first I want to understand the baseline.

## 5. A note about `contamination`

If I start with `contamination=0.01`, I'm using a provisional threshold setting that corresponds to an expected outlier proportion of about 1% on the training data. This does **not** prove that 1% of my transactions are genuinely anomalous. I should treat it as an initial assumption and inspect the results.

Because I'm training two models separately, each model applies its threshold to its own group. Their anomaly scores and thresholds should not automatically be treated as directly comparable.

## My takeaway

I used `.abs()` to create `amount_magnitude` because I want the feature to represent transaction size without the debit/credit sign. I chose two Isolation Forest models because the credit and debit magnitude distributions are very different in my dataset. This gives me a clear first baseline: find unusual transaction sizes within credits and within debits, then inspect the flagged transactions before deciding what features to add next.


In [94]:
credit_transactions = transactions.loc[transactions["transaction_type"] == "credit"].copy()

credit_features = credit_transactions[["amount_magnitude"]]

print(credit_features.shape[0])

credit_features.head()

5070


,amount_magnitude
4,57747.01
14,52037.99
20,50502.66
22,60786.74
23,57039.45


In [95]:
from sklearn.ensemble import IsolationForest

credit_model = IsolationForest(n_estimators = 100, contamination= 0.01, random_state = 42)

credit_model.fit(credit_features)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [96]:
credit_transactions["anomaly_prediction"] = credit_model.predict(credit_features)

credit_transactions["anomaly_score"] = credit_model.decision_function(credit_features)

credit_transactions.head()


,transaction_id,customer_id,date,description,amount,category,transaction_type,merchant,currency,validation_issues,year,month,day_of_week,is_weekend,amount_magnitude,anomaly_prediction,anomaly_score
4,TXN0000005,CUST0001,2025-02-05,SALARY CREDIT Employer Payroll,57747.01,Income,credit,Employer Payroll,INR,NaN,2025,2,Wednesday,False,57747.01,1,0.250024
14,TXN0000015,CUST0001,2025-07-10,SALARY CREDIT Employer Payroll,52037.99,Income,credit,Employer Payroll,INR,NaN,2025,7,Thursday,False,52037.99,1,0.228995
20,TXN0000022,CUST0001,2025-04-11,SALARY CREDIT Employer Payroll,50502.66,Income,credit,Employer Payroll,INR,NaN,2025,4,Friday,False,50502.66,1,0.227870
22,TXN0000024,CUST0001,2025-02-12,SALARY CREDIT Employer Payroll,60786.74,Income,credit,Employer Payroll,INR,NaN,2025,2,Wednesday,False,60786.74,1,0.234525
23,TXN0000025,CUST0001,2025-04-12,SALARY CREDIT Employer Payroll,57039.45,Income,credit,Employer Payroll,INR,NaN,2025,4,Saturday,True,57039.45,1,0.252184


In [97]:
credit_transactions["anomaly_flag"] = credit_transactions["anomaly_prediction"].map({1 : "normal", -1 : "anomaly"})

display(
    credit_transactions[
    ["transaction_id", "amount", "amount_magnitude",
     "anomaly_prediction", "anomaly_score", "anomaly_flag"]].head(10)
)



,transaction_id,amount,amount_magnitude,anomaly_prediction,anomaly_score,anomaly_flag
4,TXN0000005,57747.01,57747.01,1,0.250024,normal
14,TXN0000015,52037.99,52037.99,1,0.228995,normal
20,TXN0000022,50502.66,50502.66,1,0.227870,normal
22,TXN0000024,60786.74,60786.74,1,0.234525,normal
23,TXN0000025,57039.45,57039.45,1,0.252184,normal
27,TXN0000029,54296.81,54296.81,1,0.239530,normal
29,TXN0000031,56351.76,56351.76,1,0.247130,normal
32,TXN0000034,54197.95,54197.95,1,0.241704,normal
37,TXN0000039,54803.38,54803.38,1,0.248428,normal
40,TXN0000042,56986.84,56986.84,1,0.248367,normal


In [98]:
debit_transactions = transactions.loc[transactions["transaction_type"] == "debit"].copy()

debit_features = debit_transactions[["amount_magnitude"]]

debit_model = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)

debit_model.fit(debit_features)


,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [99]:
debit_transactions["anomaly_prediction"] = debit_model.predict(debit_features)

debit_transactions["anomaly_score"] = debit_model.decision_function(debit_features)

debit_transactions["anomaly_flag"] = debit_transactions["anomaly_prediction"].map({1: "normal", -1: "anomaly"})

debit_transactions[
    ["transaction_id",
      "amount", "amount_magnitude",
        "anomaly_prediction", "anomaly_score", "anomaly_flag"
     ]].head(10)

,transaction_id,amount,amount_magnitude,anomaly_prediction,anomaly_score,anomaly_flag
0,TXN0000001,-21812.91,21812.91,1,0.162224,normal
1,TXN0000002,-3611.60,3611.60,1,0.313108,normal
2,TXN0000003,-1785.61,1785.61,1,0.350483,normal
3,TXN0000004,-1318.53,1318.53,1,0.359646,normal
5,TXN0000006,-4996.62,4996.62,1,0.276314,normal
6,TXN0000007,-1766.67,1766.67,1,0.351966,normal
7,TXN0000008,-18967.39,18967.39,1,0.172129,normal
8,TXN0000009,-1284.24,1284.24,1,0.359136,normal
9,TXN0000010,-1746.26,1746.26,1,0.351917,normal
10,TXN0000011,-32157.82,32157.82,1,0.117320,normal


In [100]:
display(credit_transactions["anomaly_flag"].value_counts())

display(debit_transactions["anomaly_flag"].value_counts())

anomaly_flag
normal     5019
anomaly      51
Name: count, dtype: int64

anomaly_flag
normal     27279
anomaly       90
Name: count, dtype: int64

In [101]:
scored_transactions = pd.concat([credit_transactions, debit_transactions]).sort_index()

print("total_transactions: ", len(scored_transactions))

total_transactions:  32439


In [102]:
print(scored_transactions.groupby(["transaction_type", "anomaly_flag"]).size().unstack())

anomaly_flag      anomaly  normal
transaction_type                 
credit                 51    5019
debit                  90   27279


In [103]:
scored_transactions.to_csv("scored_transactions.csv", index = False)

In [104]:
anomalies = scored_transactions[scored_transactions["anomaly_flag"] == "anomaly"].sort_values("anomaly_score")

anomalies.head(20)

,transaction_id,customer_id,date,description,amount,category,transaction_type,merchant,currency,validation_issues,year,month,day_of_week,is_weekend,amount_magnitude,anomaly_prediction,anomaly_score,anomaly_flag
8916,TXN0008920,CUST0166,2026-06-04,SALARY CREDIT Employer Payroll,142866.31,Income,credit,Employer Payroll,INR,NaN,2026,6,Thursday,False,142866.31,-1,-0.083706,anomaly
8882,TXN0008886,CUST0166,2025-07-08,SALARY CREDIT Employer Payroll,136996.76,Income,credit,Employer Payroll,INR,NaN,2025,7,Tuesday,False,136996.76,-1,-0.079108,anomaly
29664,TXN0029668,CUST0550,2026-06-06,SALARY CREDIT Employer Payroll,135829.19,Income,credit,Employer Payroll,INR,NaN,2026,6,Saturday,True,135829.19,-1,-0.079108,anomaly
8905,TXN0008909,CUST0166,2026-08-02,SALARY CREDIT Employer Payroll,134017.20,Income,credit,Employer Payroll,INR,NaN,2026,8,Sunday,True,134017.20,-1,-0.075956,anomaly
29625,TXN0029629,CUST0550,2025-03-07,SALARY CREDIT Employer Payroll,129459.36,Income,credit,Employer Payroll,INR,NaN,2025,3,Friday,False,129459.36,-1,-0.073861,anomaly
29643,TXN0029647,CUST0550,2025-06-12,SALARY CREDIT Employer Payroll,130700.41,Income,credit,Employer Payroll,INR,NaN,2025,6,Thursday,False,130700.41,-1,-0.073861,anomaly
8876,TXN0008880,CUST0166,2025-02-04,SALARY CREDIT Employer Payroll,129089.02,Income,credit,Employer Payroll,INR,NaN,2025,2,Tuesday,False,129089.02,-1,-0.072294,anomaly
8891,TXN0008895,CUST0166,2025-05-10,SALARY CREDIT Employer Payroll,128138.84,Income,credit,Employer Payroll,INR,NaN,2025,5,Saturday,True,128138.84,-1,-0.070209,anomaly
8886,TXN0008890,CUST0166,2025-05-09,SALARY CREDIT Employer Payroll,128488.61,Income,credit,Employer Payroll,INR,NaN,2025,5,Friday,False,128488.61,-1,-0.070209,anomaly
29666,TXN0029670,CUST0550,2026-04-07,SALARY CREDIT Employer Payroll,117694.62,Income,credit,Employer Payroll,INR,NaN,2026,4,Tuesday,False,117694.62,-1,-0.053225,anomaly


In [105]:
customer_history = scored_transactions[
    (scored_transactions["customer_id"] == "CUST0098") &
    (scored_transactions["transaction_type"] == "credit")
    ].sort_values("date")

customer_history[["date", "description", "amount", "amount_magnitude", "anomaly_score", "anomaly_flag"]]

,date,description,amount,amount_magnitude,anomaly_score,anomaly_flag
5179,2025-03-02,SALARY CREDIT Employer Payroll,96856.89,96856.89,0.120546,normal
5180,2025-04-02,SALARY CREDIT Employer Payroll,105755.01,105755.01,0.007877,normal
5194,2025-04-09,SALARY CREDIT Employer Payroll,105677.46,105677.46,0.009278,normal
5199,2025-05-11,SALARY CREDIT Employer Payroll,109993.08,109993.08,-0.026135,anomaly
5187,2025-06-05,SALARY CREDIT Employer Payroll,113150.39,113150.39,-0.042612,anomaly
5195,2025-08-09,SALARY CREDIT Employer Payroll,110848.03,110848.03,-0.033036,anomaly
5202,2026-01-06,SALARY CREDIT Employer Payroll,102096.97,102096.97,0.050680,normal
5219,2026-04-05,SALARY CREDIT Employer Payroll,102886.63,102886.63,0.036867,normal
5220,2026-06-05,SALARY CREDIT Employer Payroll,106610.30,106610.30,0.001771,normal
5227,2026-09-02,SALARY CREDIT Employer Payroll,111675.14,111675.14,-0.038609,anomaly


In [106]:
customer_history["amount"].describe()

count        10.000000
mean     106554.990000
std        5041.636965
min       96856.890000
25%      103584.337500
50%      106182.655000
75%      110634.292500
max      113150.390000
Name: amount, dtype: float64

In [107]:
%whos

Variable                 Type               Data/Info
-----------------------------------------------------
IsolationForest          ABCMeta            <class 'sklearn.ensemble.<...>iforest.IsolationForest'>
KMeans                   ABCMeta            <class 'sklearn.cluster._kmeans.KMeans'>
StandardScaler           type               <class 'sklearn.preproces<...>ng._data.StandardScaler'>
anomalies                DataFrame          Shape: (141, 18)
behavior_transactions    DataFrame          Shape: (32439, 15)
calculate_prior_median   function           <function calculate_prior<...>an at 0x000002093EDD7B00>
credit_features          DataFrame          Shape: (5070, 1)
credit_model             IsolationForest    IsolationForest(contamina<...>on=0.01, random_state=42)
credit_transactions      DataFrame          Shape: (5070, 18)
customer_history         DataFrame          Shape: (10, 18)
daily_amounts            DataFrame          Shape: (31182, 6)
debit_features           DataFrame    

In [108]:
behavior_transactions = transactions.copy()

behavior_transactions["date"] = pd.to_datetime(behavior_transactions["date"],errors = "coerce")

behavior_transactions = behavior_transactions.rename(columns ={"date" : "transaction_day"})

behavior_transactions = behavior_transactions.sort_values(["customer_id", "transaction_type", "transaction_day"]).reset_index(drop = True)
behavior_transactions.head()

,transaction_id,customer_id,transaction_day,description,amount,category,transaction_type,merchant,currency,validation_issues,year,month,day_of_week,is_weekend,amount_magnitude
0,TXN0000005,CUST0001,2025-02-05,SALARY CREDIT Employer Payroll,57747.01,Income,credit,Employer Payroll,INR,NaN,2025,2,Wednesday,False,57747.01
1,TXN0000024,CUST0001,2025-02-12,SALARY CREDIT Employer Payroll,60786.74,Income,credit,Employer Payroll,INR,NaN,2025,2,Wednesday,False,60786.74
2,TXN0000022,CUST0001,2025-04-11,SALARY CREDIT Employer Payroll,50502.66,Income,credit,Employer Payroll,INR,NaN,2025,4,Friday,False,50502.66
3,TXN0000025,CUST0001,2025-04-12,SALARY CREDIT Employer Payroll,57039.45,Income,credit,Employer Payroll,INR,NaN,2025,4,Saturday,True,57039.45
4,TXN0000015,CUST0001,2025-07-10,SALARY CREDIT Employer Payroll,52037.99,Income,credit,Employer Payroll,INR,NaN,2025,7,Thursday,False,52037.99


In [109]:
daily_amounts = behavior_transactions.groupby(["customer_id", "transaction_day", "transaction_type"], as_index= False)["amount_magnitude"].median()


daily_amounts = daily_amounts.sort_values(["customer_id", "transaction_type", "transaction_day"]).rename(columns = {"amount_magnitude" : "daily_median_amount"})
daily_amounts = daily_amounts.reset_index(drop = True)

daily_amounts.head(20)

,customer_id,transaction_day,transaction_type,daily_median_amount
0,CUST0001,2025-02-05,credit,57747.01
1,CUST0001,2025-02-12,credit,60786.74
2,CUST0001,2025-04-11,credit,50502.66
3,CUST0001,2025-04-12,credit,57039.45
4,CUST0001,2025-07-10,credit,52037.99
5,CUST0001,2026-01-05,credit,56351.76
6,CUST0001,2026-02-01,credit,54296.81
7,CUST0001,2026-02-05,credit,55316.09
8,CUST0001,2026-03-03,credit,54803.38
9,CUST0001,2026-06-02,credit,54197.95


In [110]:
daily_amounts.head()

,customer_id,transaction_day,transaction_type,daily_median_amount
0,CUST0001,2025-02-05,credit,57747.01
1,CUST0001,2025-02-12,credit,60786.74
2,CUST0001,2025-04-11,credit,50502.66
3,CUST0001,2025-04-12,credit,57039.45
4,CUST0001,2025-07-10,credit,52037.99


In [111]:
def calculate_prior_median(amounts):
    previous_amounts = amounts.shift(1)

    prior_medians = previous_amounts.expanding(
        min_periods=3
    ).median()

    return prior_medians

In [113]:
daily_amounts["prior_customer_median_amount"] = (
    daily_amounts
    .groupby(["customer_id", "transaction_type"])["daily_median_amount"]
    .transform(calculate_prior_median)
)

daily_amounts.head(10)

,customer_id,transaction_day,transaction_type,daily_median_amount,prior_customer_median_amount
0,CUST0001,2025-02-05,credit,57747.01,NaN
1,CUST0001,2025-02-12,credit,60786.74,NaN
2,CUST0001,2025-04-11,credit,50502.66,NaN
3,CUST0001,2025-04-12,credit,57039.45,57747.010
4,CUST0001,2025-07-10,credit,52037.99,57393.230
5,CUST0001,2026-01-05,credit,56351.76,57039.450
6,CUST0001,2026-02-01,credit,54296.81,56695.605
7,CUST0001,2026-02-05,credit,55316.09,56351.760
8,CUST0001,2026-03-03,credit,54803.38,55833.925
9,CUST0001,2026-06-02,credit,54197.95,55316.090
